# Drug Review Sentiment Classification

This example uses the [Druglib drug reviews dataset](https://archive.ics.uci.edu/dataset/461/drug+review+dataset+druglib+com) to train a model to classify drug reviews as having positive, neutral, or negative sentiment.

## Setup

To use this notebook, first make sure your virtual environment is configured correctly. From the base cnlp_transformers directory, run:

```bash
uv sync --group notebooks
```

This will create a local virtual environment at `.venv` with all the necessary dependencies to run the notebook. Select that environment for your notebook kernel.

## Dataset information 

Data Download Source:
https://archive.ics.uci.edu/dataset/461/drug+review+dataset+druglib+com

BibTeX citation:

```bibtex
@misc{drug_reviews_(druglib.com)_461,
  author       = {Kallumadi, Surya and Grer, Felix},
  title        = {{Drug Reviews (Druglib.com)}},
  year         = {2018},
  howpublished = {UCI Machine Learning Repository},
  note         = {{DOI}: https://doi.org/10.24432/C55G6J}
}
```

Important Notes:
When using this dataset, you agree that you
1) only use the data for research purposes
2) don't use the data for any commerical purposes
3) don't distribute the data to anyone else
4) cite UCI data lab and the source


In [ ]:
# Imports
import io
import os
import zipfile
from pathlib import Path

import polars as pl
import requests

from cnlpt.data import CnlpDataset, HierarchicalDataConfig
from cnlpt.data.analysis import make_preds_df
from cnlpt.data.predictions import CnlpPredictions
from cnlpt.modeling import (
    CnnModel,
    CnnModelConfig,
    HierarchicalModel,
    HierarchicalModelConfig,
    LstmModel,
    LstmModelConfig,
    ModelType,
    ProjectionModel,
    ProjectionModelConfig,
)
from cnlpt.train_system import CnlpTrainingArguments, CnlpTrainSystem

In [ ]:
# Define dataset and output directories

DATA_DIR = Path.cwd() / "dataset"
OUTPUT_DIR = DATA_DIR = Path.cwd() / "train_output"

In [ ]:
# Download and preprocess data for training
FORCE_REDOWNLOAD = False

DATASET_ZIP_URL = (
    "https://archive.ics.uci.edu/static/public/461/drug+review+dataset+druglib+com.zip"
)


def preprocess_raw_data(unprocessed_path: str):
    return pl.read_csv(unprocessed_path, separator="\t").select(
        id="",
        sentiment=pl.col("rating").map_elements(
            lambda rating: "Negative"
            if rating < 5
            else "Neutral"
            if rating < 8
            else "Positive",
            return_dtype=pl.String,
        ),
        text=(
            pl.concat_str(
                "benefitsReview",
                "sideEffectsReview",
                "commentsReview",
                separator=" ",
            )
            .str.replace_all("\n", " <cr> ")
            .str.replace_all("\r", " <cr> ")
            .str.replace_all("\t", " ")
        ),
    )


DATA_DIR.mkdir(exist_ok=True)

if not FORCE_REDOWNLOAD and all(
    p in os.listdir(DATA_DIR) for p in ("train.tsv", "dev.tsv", "test.tsv")
):
    print("Skipping download and preprocessing, data already exists on disk.")
else:
    # Download dataset
    response = requests.get(DATASET_ZIP_URL)
    response.raise_for_status()
    zip = zipfile.ZipFile(io.BytesIO(response.content))
    zip.extractall(DATA_DIR)

    raw_train_file = DATA_DIR / "drugLibTrain_raw.tsv"
    raw_test_file = DATA_DIR / "drugLibTest_raw.tsv"

    # Preprocess raw data
    preprocessed_train_data = preprocess_raw_data(raw_train_file)
    preprocessed_test_data = preprocess_raw_data(raw_test_file)

    # 90/10 split for train and dev
    preprocessed_train_data, preprocessed_dev_data = (
        preprocessed_train_data.iter_slices(int(preprocessed_train_data.shape[0] * 0.9))
    )

    # Write to tsv files
    preprocessed_train_data.write_csv(DATA_DIR / "train.tsv", separator="\t")
    preprocessed_dev_data.write_csv(DATA_DIR / "dev.tsv", separator="\t")
    preprocessed_test_data.write_csv(DATA_DIR / "test.tsv", separator="\t")

    # Delete raw data files
    raw_train_file.unlink()
    raw_test_file.unlink()

In [ ]:
# Choose a model type

# Change this to choose a different type of model!
MODEL_TYPE = ModelType.PROJ

In [ ]:
# Prepare the dataset

if MODEL_TYPE == ModelType.HIER:
    hier_data_config = HierarchicalDataConfig(
        chunk_len=16, num_chunks=8, prepend_empty_chunk=False
    )
else:
    hier_data_config = None

dataset = CnlpDataset(
    DATA_DIR,
    task_names=["sentiment"],
    hier_config=hier_data_config,
    # max_train=100,  # optionally limit how many train/eval/test instances to use
    # max_eval=100,
    # max_test=100,
)

In [ ]:
# Initialize the model

if MODEL_TYPE == ModelType.PROJ:
    model = ProjectionModel(
        ProjectionModelConfig(tasks=dataset.tasks, vocab_size=len(dataset.tokenizer))
    )
elif MODEL_TYPE == ModelType.HIER:
    model = HierarchicalModel(
        HierarchicalModelConfig(tasks=dataset.tasks, vocab_size=len(dataset.tokenizer))
    )
elif MODEL_TYPE == ModelType.CNN:
    model = CnnModel(
        CnnModelConfig(tasks=dataset.tasks, vocab_size=len(dataset.tokenizer))
    )
elif MODEL_TYPE == ModelType.LSTM:
    model = LstmModel(
        LstmModelConfig(tasks=dataset.tasks, vocab_size=len(dataset.tokenizer))
    )

In [ ]:
# Set up the train system

train_system = CnlpTrainSystem(
    model,
    dataset,
    CnlpTrainingArguments(
        output_dir=OUTPUT_DIR,
        overwrite_output_dir=True,
        do_train=True,
        do_eval=True,
        do_predict=True,
        evals_per_epoch=2,
        num_train_epochs=3,
        learning_rate=1e-5,
        metric_for_best_model="sentiment.macro_f1",
        load_best_model_at_end=True,
        save_strategy="best",
    ),
)

In [ ]:
# Train the model!

train_system.train()

In [ ]:
# Load predictions and print metrics

preds = CnlpPredictions.load_json(OUTPUT_DIR / "predictions.json")
with pl.Config(tbl_rows=len(preds.metrics)):
    display(preds.metrics_df())

In [ ]:
# Find prediction errors

analysis = make_preds_df(preds)
analysis.select("sample_id", "text", pl.col("sentiment").struct.unnest()).select(
    "sample_id",
    "text",
    label=pl.col("labels").struct.field("values"),
    prediction=pl.col("predictions").struct.field("values"),
    probabilities=pl.col("model_output")
    .struct.field("probs")
    # labels are sorted automatically by cnlpt, so we should expect them in sorted order
    .arr.to_struct(fields=sorted(["Negative", "Neutral", "Positive"])),
).unnest("probabilities").filter(pl.col("label") != pl.col("prediction"))